In [89]:
import pandas as pd
import numpy as np

data = pd.read_csv("/Users/yuxin/Documents/NYU DS/NYU-DS/2026 Fall/ML in Finance/Project/codebase/finance_term_project/data/processed/labeled_data.csv", low_memory=False)
data.drop(columns=['Unnamed: 0'], inplace = True)
data.head(5)

,id,stmt_date,HQ_city,legal_struct,ateco_sector,def_date,fs_year,asst_intang_fixed,asst_tang_fixed,asst_fixed_fin,...,profit,days_rec,ebitda,roa,roe,wc_net,margin_fin,cf_operations,label,pred_start_dt
0,520288,2011-12-31,28.0,SRL,14.0,NaN,2011,67537.0,1137566.0,1908.0,...,-81288.0,NaN,6318.0,-3.81,-28.03,496258.0,-917029.0,-849.0,0,2012-06-01
1,520288,2008-12-31,28.0,SRL,14.0,NaN,2008,256438.0,1181416.0,860.0,...,-107382.0,NaN,46088.0,-2.76,NaN,97952.0,NaN,-3881.0,0,2009-06-01
2,520288,2009-12-31,28.0,SRL,14.0,NaN,2009,194046.0,1152014.0,860.0,...,-77819.0,NaN,67611.0,-2.17,NaN,-210671.0,NaN,32618.0,0,2010-06-01
3,520288,2007-12-31,28.0,SRL,14.0,NaN,2007,126603.0,1127807.0,620.0,...,36733.0,NaN,153060.0,6.20,52.43,-317007.0,-1184970.0,80039.0,0,2008-06-01
4,520288,2010-12-31,28.0,SRL,14.0,NaN,2010,130973.0,1136731.0,946.0,...,-83593.0,NaN,3973.0,-4.08,-22.52,398756.0,-897381.0,-2104.0,0,2011-06-01


In [ ]:
# alot of bank debt details are missing
data[data['debt_bank_st'].isna()][['id', 'stmt_date', 'debt_bank_st', 'debt_bank_lt']]

,id,stmt_date,debt_bank_st,debt_bank_lt
26,2540151,2009-12-31,NaN,NaN
29,2540151,2008-12-31,NaN,NaN
31,2860120,2010-12-31,NaN,0.0
37,3020120,2010-12-31,NaN,NaN
38,3020120,2008-12-31,NaN,NaN
...,...,...,...,...
834983,80007460233,2008-12-31,NaN,0.0
834984,80007460233,2010-12-31,NaN,424000.0
834985,80007460233,2009-12-31,NaN,0.0
834987,80007460233,2011-12-31,NaN,NaN


In [91]:
#adding sector names into the dataset
industries = pd.read_csv("/Users/yuxin/Documents/NYU DS/NYU-DS/2026 Fall/ML in Finance/Project/codebase/finance_term_project/data/interim/ateco_sector.csv", low_memory=False)
industries['code'] = pd.to_numeric(industries['code'])
industries.head()

,code,industry_english
0,1,"Crop and animal production, hunting and relate..."
1,2,Forestry and logging
2,3,Fishing and aquaculture
3,5,Mining of coal and lignite
4,6,Extraction of crude petroleum and natural gas


In [4]:
groups = {
    "1 Identity & attributes": ["id", "stmt_date", "HQ_city", "legal_struct", "ateco_sector", "fs_year", "def_date", "pred_start_dt"],
    "2 Assets": ["asst_intang_fixed", "asst_tang_fixed", "asst_fixed_fin", "asst_current", "AR", "cash_and_equiv", "asst_tot", "goodwill"],
    "3 Equity & liabilities": ["eqty_tot", "eqty_corp_family_tot", "liab_lt", "liab_lt_emp", "debt_bank_st", "debt_bank_lt",
                "debt_fin_st", "debt_fin_lt", "AP_st", "AP_lt", "debt_st", "debt_lt"],
    "4 Income statement": ["rev_operating", "COGS", "prof_operations", "inc_financing", "exp_financing", "prof_financing",
              "inc_extraord", "taxes", "profit", "ebitda"],
    "5 Cash flow & derived ratios": ["cf_operations", "wc_net", "margin_fin", "days_rec", "roa", "roe"],
    "6 Label": ["label"]
}
assert sorted(sum(groups.values(), [])) == sorted(data.columns)

rows = []
for g, cols in groups.items():
    for c in cols:
        rows.append({"group": g, "col": c, "n_missing": data[c].isna().sum(),
                     "pct_missing": round(data[c].isna().mean() * 100, 2),
                     "n_zero": (data[c] == 0).sum(), "pct_zero": round((data[c] == 0).mean() * 100, 2)})
raw_miss = pd.DataFrame(rows).set_index(["group", "col"])
raw_miss

n_missing  pct_missing  \
group                        col                                            
1 Identity & attributes      id                            0         0.00   
                             stmt_date                     0         0.00   
                             HQ_city                     254         0.03   
                             legal_struct                  0         0.00   
                             ateco_sector                  0         0.00   
                             fs_year                       0         0.00   
                             def_date                 793838        95.07   
                             pred_start_dt                 0         0.00   
2 Assets                     asst_intang_fixed           162         0.02   
                             asst_tang_fixed             103         0.01   
                             asst_fixed_fin              236         0.03   
                             asst_current                  1         0.00   
                             AR                           12         0.00   
                             cash_and_equiv               17         0.00   
                             asst_tot                      0         0.00   
                             goodwill                     21         0.00   
3 Equity & liabilities       eqty_tot                      1         0.00   
                             eqty_corp_family_tot     835016       100.00   
                             liab_lt                     294         0.04   
                             liab_lt_emp                 162         0.02   
                             debt_bank_st             105276        12.61   
                             debt_bank_lt              78120         9.36   
                             debt_fin_st              106709        12.78   
                             debt_fin_lt               79210         9.49   
                             AP_st                    104581        12.52   
                             AP_lt                     79386         9.51   
                             debt_st                       5         0.00   
                             debt_lt                     154         0.02   
4 Income statement           rev_operating               174         0.02   
                             COGS                         24         0.00   
                             prof_operations              21         0.00   
                             inc_financing                26         0.00   
                             exp_financing               366         0.04   
                             prof_financing              655         0.08   
                             inc_extraord                305         0.04   
                             taxes                        32         0.00   
                             profit                       20         0.00   
                             ebitda                       81         0.01   
5 Cash flow & derived ratios cf_operations                88         0.01   
                             wc_net                        6         0.00   
                             margin_fin                31278         3.75   
                             days_rec                 600807        71.95   
                             roa                          27         0.00   
                             roe                       56624         6.78   
6 Label                      label                         0         0.00   

                                                   n_zero  pct_zero  
group                        col                                     
1 Identity & attributes      id                         0      0.00  
                             stmt_date                  0      0.00  
                             HQ_city                    0      0.00  
                             legal_struct               0      0.00  
           

### Grouping features

In [ ]:
# from sklearn.model_selection import StratifiedKFold

# def fit_oof_target_encoder(df, column="ateco_sector", target="label",
#                            n_splits=5, smoothing=20, random_state=42):
#     """
#     Returns:
#       encoded_train: leakage-safe, out-of-fold target encoding for df[column]
#       full_mapping: mapping fit on all training rows; use this for validation/test data
#       global_mean: fallback value for unseen or missing categories
#     """
#     global_mean = df[target].mean()
#     encoded_train = pd.Series(index=df.index, dtype=float)

#     folds = StratifiedKFold(
#         n_splits=n_splits,
#         shuffle=True,
#         random_state=random_state,
#     )

#     for train_idx, valid_idx in folds.split(df, df[target]):
#         fold_train = df.iloc[train_idx]
#         fold_mean = fold_train[target].mean()

#         stats = fold_train.groupby(column)[target].agg(["sum", "count"])

#         # Smoothed category default rate:
#         # small categories are pulled toward the overall default rate.
#         mapping = (stats["sum"] + smoothing * fold_mean) / (
#             stats["count"] + smoothing
#         )

#         encoded_train.iloc[valid_idx] = (
#             df.iloc[valid_idx][column]
#             .map(mapping)
#             .fillna(fold_mean)  # unseen/missing category
#         )

#     # Fit one final mapping on all training data for validation/test transformation.
#     full_stats = df.groupby(column)[target].agg(["sum", "count"])
#     full_mapping = (full_stats["sum"] + smoothing * global_mean) / (
#         full_stats["count"] + smoothing
#     )

#     return encoded_train, full_mapping, global_mean


# def transform_target_encoder(df, column, mapping, global_mean):
#     """Apply a mapping learned from training data only."""
#     return df[column].map(mapping).fillna(global_mean)

In [15]:
# Missing-value handling; see docs/project-update/features/02-missing-value.md
DROP_COLS = ["Unnamed: 0", "eqty_corp_family_tot", "days_rec", "roe"]
NON_NUMERIC = ["stmt_date", "legal_struct", "def_date", "pred_start_dt"]
DEBT_DETAIL = ["debt_bank_st", "debt_bank_lt", "debt_fin_st", "debt_fin_lt", "AP_st", "AP_lt"]
# Method B: columns filled with (sector median of column / asst_tot) * asst_tot
B_COLS = [
    "asst_intang_fixed", "asst_tang_fixed", "asst_fixed_fin", "asst_current", "AR", "cash_and_equiv", "goodwill",
    "eqty_tot", "liab_lt", "liab_lt_emp", *DEBT_DETAIL, "debt_st", "debt_lt",
    "rev_operating", "COGS", "prof_operations", "inc_financing", "exp_financing", "prof_financing",
    "inc_extraord", "taxes", "profit", "ebitda", "cf_operations",
]


def sector_medians(train):
    """Median table for method B, computed from the training data: indexed by ateco_sector, last row "ALL" is the overall median.
    The asst_tot column holds the median amount (safeguard for holdout rows missing asst_tot); the other columns hold median ratios to asst_tot."""
    train = _to_numeric(train)
    ratio = train[B_COLS].div(train["asst_tot"], axis=0)
    ratio["asst_tot"] = train["asst_tot"]
    med = ratio.groupby(train["ateco_sector"]).median()
    med.loc["ALL"] = ratio.median()
    return med


def _to_numeric(df):
    # Blanks, "NA" and other missing-value spellings all become NaN
    df = df.copy()
    for c in df.columns.difference(NON_NUMERIC):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def fill_missing(df, medians):
    """Take raw data, return filled data. medians must be the result of sector_medians(training data)."""
    df = _to_numeric(df).drop(columns=DROP_COLS, errors="ignore")

    # Method A: a missing HQ_city becomes its own category
    df["HQ_city"] = df["HQ_city"].map("{:.0f}".format, na_action="ignore").fillna("Unknown")

    # Method C: record the flag before filling
    df["debt_detail_missing"] = df[DEBT_DETAIL].isna().any(axis=1).astype(int)

    # Method D: prof_operations = rev_operating - COGS; restore whichever of the three is missing
    df["rev_operating"] = df["rev_operating"].fillna(df["prof_operations"] + df["COGS"])
    df["COGS"] = df["COGS"].fillna(df["rev_operating"] - df["prof_operations"])
    df["prof_operations"] = df["prof_operations"].fillna(df["rev_operating"] - df["COGS"])

    # Method B: look up the sector medians; sectors absent from training use the overall median
    m = medians.drop(index="ALL").reindex(df["ateco_sector"]).set_axis(df.index).fillna(medians.loc["ALL"])
    df["asst_tot"] = df["asst_tot"].fillna(m["asst_tot"])
    df[B_COLS] = df[B_COLS].fillna(m[B_COLS].mul(df["asst_tot"], axis=0))

    # Method E: restore by formula (after B and D, so every input is already filled)
    fixed = df["asst_intang_fixed"] + df["asst_tang_fixed"] + df["asst_fixed_fin"]
    df["margin_fin"] = df["margin_fin"].fillna(df["eqty_tot"] - fixed)
    df["wc_net"] = df["wc_net"].fillna(df["asst_current"] - df["debt_st"])
    df["roa"] = df["roa"].fillna(df["prof_operations"] / df["asst_tot"] * 100)

    return df


In [ ]:
# medians = sector_medians(data)
# filled = fill_missing(data, medians)

# # Raw-input checks: def_date is the only field allowed to remain missing.
# assert filled.drop(columns="def_date").isna().sum().sum() == 0
# assert filled["pred_start_dt"].notna().all()
# num = data.columns.difference(DROP_COLS + NON_NUMERIC + ["HQ_city"])
# orig = data[num].notna()
# assert (filled[num][orig] == data[num][orig]).sum().sum() == orig.sum().sum()
# assert filled["debt_detail_missing"].sum() == data[DEBT_DETAIL].isna().any(axis=1).sum()
# assert (filled["HQ_city"] == "Unknown").sum() == data["HQ_city"].isna().sum()

# # Engineered-feature checks: their missingness is intentionally reported in `miss` above.
# assert set(ENGINEERED_COLS).issubset(mg_data.columns)
# assert mg_data["past_num_defaults"].ge(0).all()

# # Check: missing-value spellings a holdout may use, and a sector never seen in training
# raw = data.head(3).astype(str)
# raw.loc[raw.index[0], ["AR", "HQ_city", "asst_tot"]] = ["", " ", "NA"]
# raw.loc[raw.index[1], "ateco_sector"] = "999"
# out = fill_missing(raw, medians)
# assert out.drop(columns="def_date").isna().sum().sum() == 0
# assert out["HQ_city"].iloc[0] == "Unknown"
# filled.shape


(835016, 43)